# The SAM 3 prompts of every VSI-Bench question

On Colab's CPU (no GPU needed), a few minutes. It unpacks the code, downloads VSI-Bench's `test.jsonl` (5,130
questions, checked by SHA-256), installs spaCy 3.8 with `en_core_web_md` 3.8.0 and NLTK (the versions
`hpc/setup_env.sh` installs), runs `extract_prompts.py`, and saves `prompts_full.jsonl` to Drive.
`build_gold_sheets.py` then makes the gold sheets from it with numpy only.

**File on Google Drive**, in `MyDrive/mvp/`: `mvp-3d-vlm-prompts.zip`.

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/mvp"
CODE_ZIP = f"{DRIVE_DIR}/mvp-3d-vlm-prompts.zip"
OUT = f"{DRIVE_DIR}/prompts_full.jsonl"
VSI_REVISION = "3a6a1cf9ac54c403509b15260897cc6ddc41b408"
VSI_SHA256 = "aa3ff8cc2010e555f4f865ff96483812ec0b275005b68562f53bb114ad4db9db"

import hashlib, shutil, subprocess, sys, zipfile
from pathlib import Path
from google.colab import drive
from huggingface_hub import hf_hub_download

drive.mount("/content/drive")
assert Path(CODE_ZIP).is_file(), f"missing {CODE_ZIP}: upload it to {DRIVE_DIR}"
shutil.rmtree("/content/mvp-3d-vlm", ignore_errors=True)
zipfile.ZipFile(CODE_ZIP).extractall("/content")
questions = hf_hub_download("nyu-visionx/VSI-Bench", "test.jsonl", repo_type="dataset", revision=VSI_REVISION)
assert hashlib.sha256(Path(questions).read_bytes()).hexdigest() == VSI_SHA256, "not the expected test.jsonl"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "spacy>=3.8,<3.9", "nltk",
                "en_core_web_md @ https://github.com/explosion/spacy-models/releases/download/en_core_web_md-3.8.0/"
                "en_core_web_md-3.8.0-py3-none-any.whl"], check=True)
print("ready:", questions)

In [ ]:
E = "/content/mvp-3d-vlm/experiments/2026-11-19-gold-sheet-relations"
r = subprocess.run([sys.executable, f"{E}/extract_prompts.py", "--questions", questions, "--out", "/content/prompts_full.jsonl"],
                   capture_output=True, text=True)
print(r.stdout[-1500:])
assert r.returncode == 0, r.stderr[-3000:]
shutil.copy("/content/prompts_full.jsonl", OUT)
import spacy, en_core_web_md, nltk
print(f"spacy {spacy.__version__}, en_core_web_md {en_core_web_md.__version__}, nltk {nltk.__version__}")
print(f"saved {OUT}, sha256 {hashlib.sha256(Path(OUT).read_bytes()).hexdigest()}")